[◀ Notebook 15: Concurrency & Async](15_concurrency_and_async.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 17: Modern Packaging ▶](17_modern_packaging.ipynb)

# Notebook 16: Modules and Packages

Welcome to Notebook 16 of the Bottom-Up Python Tutorial. In this notebook, we examine how Python structures code across multiple files and directories. We will go under the hood of Python's import system, explain the search path mechanics, look at regular vs. namespace packages, demystify circular imports, and learn how to write an executable package with a CLI interface.

### References:
- [Python Language Reference: The Import System](https://docs.python.org/3/reference/import.html)
- [PEP 328: Imports: Multi-Line and Absolute/Relative](https://peps.python.org/pep-0328/)
- [PEP 420: Implicit Namespace Packages](https://peps.python.org/pep-0420/)

## 1. The Import Mechanism under the Hood

When you run `import foo`, Python does not simply read the file. It executes a multi-step search and loading process:

1. **Check the Cache (`sys.modules`):** Python first looks up `foo` in `sys.modules`, which is a dictionary mapping module names to loaded module objects. If it's already in there, Python returns it immediately. This makes imports extremely fast if done repeatedly.
2. **Finders and Loaders (`sys.meta_path`):** If the module is not cached, Python queries a list of finders (`sys.meta_path`). Finders inspect the filesystem (or other sources) to see if they can find the module. Once found, they return a module spec containing the loader.
3. **Execution:** The loader compiles and executes the module's source code in a new module namespace. The newly created module is then added to `sys.modules`.

### The Search Path (`sys.path`)
Python looks for modules in the directories listed in `sys.path`. The directories are searched in the following order:
1. The directory containing the input script (or the current directory if running interactively).
2. The environment variable `PYTHONPATH` (if set).
3. Installation-dependent default directories (standard library and site-packages).

In [ ]:
import sys

# Look at sys.path to see where Python searches for modules
print("Module Search Path (sys.path):")
for path in sys.path[:5]:  # print first 5 paths for readability
    print(f"  {path}")

# Check if 'math' module is cached in sys.modules
print("Is 'math' in sys.modules?", "math" in sys.modules)
import math
print("Is 'math' in sys.modules now?", "math" in sys.modules)

## 2. Absolute vs. Relative Imports

- **Absolute Imports:** Specify the full path from the project's root folder or the standard library. E.g., `import myproject.utils.helpers`.
- **Relative Imports:** Use leading dots to specify a module relative to the current module's position. A single dot `.` refers to the current package; two dots `..` refer to the parent package. E.g., `from .sibling import foo`.

> [!IMPORTANT]
> Relative imports are only valid within packages. If you execute a file containing relative imports directly (e.g. `python my_file.py`), Python will raise `ImportError: attempted relative import with no known parent package`. This is because relative imports rely on the module's `__name__` and `__package__` attributes, which are not set when executed directly.

## 3. Regular Packages vs. Namespace Packages

- **Regular Packages:** A directory containing an `__init__.py` file. The existence of `__init__.py` marks the directory as a Python package. When the package is imported, `__init__.py` is executed, initializing the package namespace. It is also used to control exports via the special `__all__` list.
- **Namespace Packages (PEP 420):** Introduced in Python 3.3. These allow developers to distribute a single package across multiple separate directories or distribution archives. Namespace packages *must not* contain an `__init__.py` file.

## 4. Making a Package Executable (`__main__.py`) and Handling CLI Arguments

If a package directory contains a `__main__.py` file, you can execute the package as a script from the command line using the `-m` switch:
```bash
python -m package_name
```
This runs the code in `__main__.py` within the package context, making it extremely easy to build CLI applications.

### Command-Line Arguments: The Mechanics of `sys.argv`

When a Python script or package is executed from the terminal, the operating system passes the command-line arguments to the Python runtime. Python captures these arguments and exposes them via the `sys.argv` list in the `sys` module.

#### 1. CPython Runtime Initialization of `sys.argv`
During the CPython startup phase (specifically within `pymain_run_python` and related initialization routines in `Python/pylifecycle.c` or `Modules/main.c`), the interpreter accesses the command-line arguments passed via the standard C arguments `int argc` and `wchar_t **argv`.
- CPython parses these system-level wide character arrays and builds a Python list of strings.
- **`sys.argv[0]`**: Represents the execution path of the invoked script. If you execute a package with `python -m package_name`, Python sets `sys.argv[0]` to the absolute filesystem path of the package's `__main__.py` file.
- **`sys.argv[1:]`**: Represents the positional command-line arguments passed *after* the module/script flag.

#### 2. The String Constraints of Command-Line Arguments
Because the operating system passes all CLI arguments as a character array, **every element in `sys.argv` is always a string (`str`)**.
- Python does *not* inspect the contents to guess the type (e.g., if a user inputs `42`, it is received as the string `"42"`).
- Since Python is a dynamically but **strongly typed** language, mixing types or expecting automatic coercion leads to bugs or runtime exceptions:
  - Performing `+` on `"10"` and `"5"` returns the concatenated string `"105"`, not the integer `15`.
  - Performing arithmetic subtraction (`-`) or multiplication/division on string objects raises a `TypeError`.

#### 3. Explicit Type Casting and Error Handling
To use CLI arguments for mathematical or logical operations, you must explicitly cast them using constructors like `int()`, `float()`, or validate them. Because command-line input is external and untrusted, you should always handle two primary exception types:
- **`IndexError`**: Occurs if you attempt to access an index (e.g., `sys.argv[2]`) that was not supplied by the user (i.e., `len(sys.argv)` is less than expected).
- **`ValueError`**: Occurs if the argument string cannot be successfully parsed into the target type (e.g., `int("abc")` raises `ValueError`).

### Concrete Example: Safe Parsing of `sys.argv`

The following code illustrates how to inspect `sys.argv` programmatically, verify the correct number of arguments, perform casting with robust error handling, and execute conditional logic:

```python
import sys

def parse_cli_args():
    # We expect: python script.py <operation> <operand1> <operand2>
    # Thus, len(sys.argv) must be 4:
    # sys.argv[0] -> script.py (or path/to/__main__.py)
    # sys.argv[1] -> operation (e.g., 'add')
    # sys.argv[2] -> operand1 (e.g., '15')
    # sys.argv[3] -> operand2 (e.g., '35')
    
    if len(sys.argv) < 4:
        print("Error: Missing arguments.", file=sys.stderr)
        print("Usage: python -m calculator_cli <add|subtract> <num1> <num2>", file=sys.stderr)
        sys.exit(1)
        
    operation = sys.argv[1]
    
    try:
        # Explicit type casting from string to integer
        val1 = int(sys.argv[2])
        val2 = int(sys.argv[3])
    except ValueError as e:
        print(f"Error: Operands must be integers. Details: {e}", file=sys.stderr)
        sys.exit(1)
        
    return operation, val1, val2
```

## 5. Circular Imports

A circular import occurs when `module_a` imports `module_b`, and `module_b` imports `module_a`. At execution time, this causes one module to be loaded in a partially initialized state, leading to `AttributeError` or `ImportError` when code tries to access names that haven't been defined yet.

### Solutions:
1. **Deferred Imports:** Move the `import` statement inside a function or method so it's only executed when the function is called, rather than at module initialization time.
2. **Refactoring:** Extract the shared dependencies into a third module `module_c` that both `module_a` and `module_b` can import.
3. **Absolute Namespaces:** Avoid importing specific variables from sibling modules at top-level; instead, import the module itself.

---

## 6. Coding Challenges

Complete the challenges below by writing python code to create the module structures on disk, and verify them using the test cells.

### Challenge 1: Diagnosing and Fixing Circular Imports

Below is code that writes a circular import bug into a temporary directory `circular_bug/`.
Try running the cell and see the error.

Your challenge is to write the code that fixes this circular import by **deferring** the import of `a` inside `b`'s functions, or vice versa, and writing the corrected files to the filesystem.

In [ ]:
import os

# Let's set up the buggy package structure first
os.makedirs("assets/circular_bug", exist_ok=True)

with open("assets/circular_bug/__init__.py", "w") as f:
    f.write("")

with open("assets/circular_bug/a.py", "w") as f:
    import sys
    if 'assets' not in sys.path: sys.path.insert(0, 'assets')
    f.write("from assets.circular_bug.b import get_b_val\n\ndef get_a_val():\n    return 42\n\ndef test_a():\n    return f'a got {get_b_val()}'\n")

with open("assets/circular_bug/b.py", "w") as f:
    import sys
    if 'assets' not in sys.path: sys.path.insert(0, 'assets')
    f.write("from assets.circular_bug.a import get_a_val\n\ndef get_b_val():\n    return 99\n\ndef test_b():\n    return f'b got {get_a_val()}'\n")

In [ ]:
# This import will fail due to circular dependencies!
try:
    # Remove from cache if loaded previously
    sys.modules.pop("assets.circular_bug.a", None)
    sys.modules.pop("assets.circular_bug.b", None)
    import sys
    if 'assets' not in sys.path: sys.path.insert(0, 'assets')
    from assets.circular_bug.a import test_a
    print(test_a())
except Exception as e:
    print(f"Import failed as expected! Error: {e}")

In [ ]:
# TODO: Rewrite 'circular_bug/b.py' (or a.py) to resolve the circular import 
# by deferring the import inside the function body.

fixed_b_content = """
# Write your fixed b.py code here
"""

# Open and write fixed_b_content to circular_bug/b.py
with open("assets/circular_bug/b.py", "w") as f:
    f.write(fixed_b_content)

In [ ]:
# Solution Code for Challenge 1
# fixed_b_content = """
# def get_b_val():
#     return 99
# 
# def test_b():
#     import sys
#     if 'assets' not in sys.path: sys.path.insert(0, 'assets')
#     from assets.circular_bug.a import get_a_val
#     return f'b got {get_a_val()}'
# """
# with open("assets/circular_bug/b.py", "w") as f:
#     f.write(fixed_b_content)


In [ ]:
# Challenge 1 Verification Test
# Let's clear the module cache to force re-importing the updated files
sys.modules.pop("assets.circular_bug", None)
sys.modules.pop("assets.circular_bug.a", None)
sys.modules.pop("assets.circular_bug.b", None)

try:
    import sys
    if 'assets' not in sys.path: sys.path.insert(0, 'assets')
    from assets.circular_bug.a import test_a
    import sys
    if 'assets' not in sys.path: sys.path.insert(0, 'assets')
    from assets.circular_bug.b import test_b
    
    res_a = test_a()
    res_b = test_b()
    
    assert res_a == "a got 99", f"Expected 'a got 99', got '{res_a}'"
    assert res_b == "b got 42", f"Expected 'b got 42', got '{res_b}'"
    print("Challenge 1: Success! Circular import resolved.")
except Exception as e:
    raise AssertionError(f"Import or call failed with fixed code: {e}")

### Challenge 2: Executable Package with CLI Entry Point

Create an executable package called `calculator_cli` on disk.
Structure it as follows:
1. `calculator_cli/` (directory)
2. `calculator_cli/__init__.py` (exposes `add` and `subtract` functions)
3. `calculator_cli/math_ops.py` (defines `add(x, y)` and `subtract(x, y)`)
4. `calculator_cli/__main__.py` (parses arguments from command line using `sys.argv` or `argparse` and calls the appropriate function. It should print only the numeric result.)

The CLI command will look like this:
```bash
python -m calculator_cli add 10 5
```
Or:
```bash
python -m calculator_cli subtract 10 5
```

In [ ]:
# TODO: Write python code to create the calculator_cli package directory and its 4 files.
import os

os.makedirs("assets/calculator_cli", exist_ok=True)

# Create math_ops.py
with open("assets/calculator_cli/math_ops.py", "w") as f:
    # Write math functions
    pass

# Create __init__.py
with open("assets/calculator_cli/__init__.py", "w") as f:
    # Export add and subtract
    pass

# Create __main__.py
with open("assets/calculator_cli/__main__.py", "w") as f:
    # Parse CLI args and print result
    pass

In [ ]:
# Solution Code for Challenge 2
# with open("assets/calculator_cli/math_ops.py", "w") as f:
#     f.write("def add(x, y): return x + y\ndef subtract(x, y): return x - y\n")
# 
# with open("assets/calculator_cli/__init__.py", "w") as f:
#     f.write("from .math_ops import add, subtract\n")
# 
# with open("assets/calculator_cli/__main__.py", "w") as f:
#     f.write("""
# import sys
# from calculator_cli import add, subtract
# if __name__ == '__main__':
#     op = sys.argv[1]
#     val1 = int(sys.argv[2])
#     val2 = int(sys.argv[3])
#     if op == 'add':
#         print(add(val1, val2))
#     elif op == 'subtract':
#         print(subtract(val1, val2))
# """)

In [ ]:
# Challenge 2 Verification Test
import subprocess
import sys

# Run calculator_cli add via subprocess
res_add = subprocess.run(
    [sys.executable, "-m", "assets.calculator_cli", "add", "15", "35"],
    capture_output=True,
    text=True
)
out_add = res_add.stdout.strip()

# Run calculator_cli subtract via subprocess
res_sub = subprocess.run(
    [sys.executable, "-m", "assets.calculator_cli", "subtract", "100", "40"],
    capture_output=True,
    text=True
)
out_sub = res_sub.stdout.strip()

print(f"CLI 'add 15 35' output: {out_add}")
print(f"CLI 'subtract 100 40' output: {out_sub}")

assert out_add == "50", f"Expected '50', got '{out_add}'"
assert out_sub == "60", f"Expected '60', got '{out_sub}'"
print("Challenge 2: Success! Executable package and CLI work correctly.")

[◀ Notebook 15: Concurrency & Async](15_concurrency_and_async.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 17: Modern Packaging ▶](17_modern_packaging.ipynb)